# Quickstart: the analysis-ready panel

This notebook loads the panel with `load_panel()`, then rebuilds two of the page's headline figures from it with
plain pandas, and checks each against `results.json`:

1. **The recovery curve** (chapter 4): each storm's share of peak-report sites restored, by day since its peak, and
   the median time to 50% restored across storms.
2. **The cause shares** (chapter 5): damage, transport, power, unattributed and over-attributed sites as shares of
   sites out, all storms pooled.

The definitions follow `code/src/10_analysis.py`; the bootstrap intervals are not recomputed here. CI runs this
notebook on every push (`pytest --nbmake examples/quickstart.ipynb`).

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

HERE = Path.cwd()
REPO = next(d for d in [HERE, *HERE.parents] if (d / "code" / "src" / "load.py").exists() or (d / "src" / "load.py").exists())
SRC = REPO / "code" / "src" if (REPO / "code" / "src").exists() else REPO / "src"
sys.path.insert(0, str(SRC))
from load import clean_rows, load_panel  # noqa: E402

results_path = next(p for p in (REPO / "data" / "analysis" / "results.json", REPO / "output" / "results.json") if p.exists())
results = json.loads(results_path.read_text(encoding="utf-8"))
panel = load_panel()
print(f"{len(panel):,} county-report rows, {panel.storm_key.nunique()} storms, {panel.doc_id.nunique()} reports")
panel[["storm", "report_date", "state", "county", "sites_served", "sites_out", "days_since_day0"]].head()

## 1. The recovery curve

For each storm: the **peak** is the report with the most sites out (ties: the earliest). The counties listed in it
are followed through every later report; a county that leaves the reports keeps its last value (it is not counted as
restored). Restored share = 1 − sites out / sites out at the peak. A storm reaches 50% at the first report from which it
stays at or above 50% in every later report; a storm that never does is censored at its last report. Time-based
results use the `standard` rows (`clean_rows`).

In [ ]:
def recovery_curve(p):
    reports = (p.drop_duplicates("doc_id")[["doc_id", "days_since_day0"]]
               .sort_values(["days_since_day0", "doc_id"]).reset_index(drop=True))
    totals = p.groupby("doc_id").sites_out.sum().reindex(reports.doc_id)
    peak = totals.to_numpy().argmax()  # first maximum in time order
    reports = reports.iloc[peak:].reset_index(drop=True)
    counties = sorted(p[p.doc_id == reports.doc_id[0]].fips.unique())
    out = (p[p.fips.isin(counties)].pivot_table(index="fips", columns="doc_id", values="sites_out", aggfunc="sum")
           .reindex(index=counties, columns=reports.doc_id).ffill(axis=1).sum())
    return pd.DataFrame({"days_since_peak": (reports.days_since_day0 - reports.days_since_day0[0]).to_numpy(),
                         "restored_share": (1 - out / out.iloc[0]).to_numpy()})


def time_to(curve, threshold):
    past = (curve.restored_share >= threshold).to_numpy()
    stays = np.logical_and.accumulate(past[::-1])[::-1]
    if not stays.any():
        return curve.days_since_peak.iloc[-1], True  # censored at the last report
    return curve.days_since_peak.iloc[stays.argmax()], False


timed = clean_rows(panel, "standard")
curves, t50 = {}, {}
for key, p in timed.groupby("storm_key"):
    c = recovery_curve(p)
    curves[key] = (p.storm.iloc[0], c)
    if len(c) > 1:  # storms with no report after the peak are left out of time summaries
        t50[key] = time_to(c, 0.5)
print(f"{len(t50)} storms with a report after the peak; {len(curves) - len(t50)} without")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
for key, (storm, c) in curves.items():
    ax.step(c.days_since_peak, 100 * c.restored_share, where="post", alpha=0.6, lw=1.2)
ax.axhline(50, color="0.4", lw=0.8, ls="--")
ax.set(xlabel="Days since the storm's peak report", ylabel="Peak-report sites restored (%)", xlim=(0, 14),
       title="Recovery curves, one line per storm (counties leaving the reports stay out)")
plt.show()

In [ ]:
def median_with_censoring(values, censored):
    """The median across storms, only when censoring cannot move it (else None)."""
    v, c = np.asarray(values, float), np.asarray(censored, bool)
    low, high = np.median(v), np.median(np.where(c, np.inf, v))
    return float(low) if np.isfinite(high) and low == high else None


values, censored = zip(*t50.values())
ours = median_with_censoring(values, censored)
published = results["chapter_4"]["headlines"]["median_time_to_50pct_restored"]
print(f"median days from peak to 50% restored: {ours} (results.json: {published['value']}, n = {published['n']})")
assert ours == published["value"] and len(t50) == published["n"]

by_storm = {s["storm_key"]: s["time_to_50pct_restored"] for s in results["chapter_4"]["storms"]}
for key, (t, cens) in t50.items():
    want = by_storm[key]
    assert cens == want["censored"] and np.isclose(t, want["censored_at"] if cens else want["value"], rtol=1e-8), key
print("every storm's time to 50% matches results.json")

## 2. The cause shares

The FCC reports split sites out into damage, transport (backhaul) and power. Over the rows that print all three, per
report: unattributed = sites out not given a cause, over-attributed = causes beyond sites out (a site can be out for
more than one reason). All shares are of sites out, pooled over every report with sites out of every storm that
prints causes. Over-attribution is shown beside the others, never netted. This chapter uses every row.

In [ ]:
SHARES = {"damage": "out_damage", "transport": "out_transport", "power": "out_power",
          "unattributed": "out_unattributed", "overattributed": "out_overattributed"}
with_causes = panel[panel[["out_damage", "out_transport", "out_power"]].notna().all(axis=1)]
by_report = with_causes.groupby(["storm_key", "doc_id"])[["sites_out", *SHARES.values()]].sum()
by_report = by_report[by_report.sites_out > 0]
total = by_report.sum()
shares = pd.Series({k: 100 * total[col] / total.sites_out for k, col in SHARES.items()}, name="% of sites out")
published = results["chapter_5"]["headlines"]["pooled_all_storms"]
check = pd.DataFrame({"ours": shares, "results.json": {k: published[k]["value"] for k in SHARES}})
print(f"{by_report.index.get_level_values('storm_key').nunique()} storms, {len(by_report)} reports with causes")
assert np.allclose(check.ours, check["results.json"], rtol=1e-8)
check.round(2)

In [ ]:
ax = shares.plot.barh(figsize=(7, 3), color=["#4c72b0", "#55a868", "#c44e52", "#8c8c8c", "#dd8452"])
ax.invert_yaxis()
ax.set(xlabel="% of sites out, all storms pooled", title="Why cell sites went dark (shares of sites out)")
plt.show()